In [ ]:
import pandas as pd
from collections import Counter

df = pd.read_json("/workspace/daeyong/ideal_steps/musique_redundancy_rewriting.json")
print(len(df))
# for i, row in df.iterrows():
#     print("Question:", row["question"])
#     print("Steps:", row['corrupted_steps'])
#     print('------------------------')
Counter(df['corrupted_step_index'])

In [ ]:
x = pd.read_json("/workspace/daeyong/ideal_steps/musique_unsupported.json")
total = pd.read_json("/workspace/daeyong/ideal_steps/hotpotqa_ideal_steps_passage_mapped.json")

print(len(set(x['question'])))
print(len(set(total['question'])))

In [ ]:
con = pd.read_json("/workspace/daeyong/ideal_steps/musique_ideal_steps.json")

count = 0
no_count = 0
for i, row in con.iterrows():
    for step in row['ideal_steps']:
        if not step.endswith("(Attribution)"):
            continue
        
        if step.split(":")[1].strip().startswith("According to Passage "):
            count += 1
        else:
            no_count += 1
            print(step)

print(f"Count with citation: {count}")
print(f"Count without citation: {no_count}")

In [ ]:
import pandas as pd

filter_list = [
    "do not mention", "does not mention", "do not provide", "does not provide",
    "do not state", "does not state", "not provided", "not mentioned", "not stated", "not explicitly mentioned",
    "not explicitly provided", "not explicitly stated", "not present in",
    "no information in the provided", "not explicitly state", "not explicitly mention",
    "does not specify", "is not specified", "there is no explicit information", "there is no information",
    "is not directly mentioned", "is not directly stated", "is not directly provided", "there's no direct information"
]

# Load dataframe
x = pd.read_json("/workspace/daeyong/ideal_steps/2wiki_redundancy_rewriting.json")
print("Total rows:", len(x))

# Filter rows where none of the phrases appear in any ideal_steps entry
filtered_df = x[
    x['ideal_steps'].apply(
        lambda steps: not any(
            any(phrase in step for phrase in filter_list) for step in steps
        )
    )
]

print("Rows with NO filter phrases:", len(filtered_df))

# Save result
filtered_df.to_json("/workspace/daeyong/ideal_steps/2wiki_redundancy_rewriting_filtered.json", orient='records', indent=2)

In [ ]:
import os
import pandas as pd

folder_path = "/workspace/daeyong/finetuning_data"

filter_list = [
	"do not mention", "does not mention", "do not provide", "does not provide",
	"do not state", "does not state", "not provided", "not mentioned", "not stated", 
	"not explicitly mentioned", "not explicitly provided", "not explicitly stated", 
	"not present in", "no information in the provided", "not explicitly state", 
	"not explicitly mention", "does not specify", "is not specified", 
	"there is no explicit information", "there is no information",
	"is not directly mentioned", "is not directly stated", "is not directly provided", 
	"there's no direct information"
]

json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]

print(f"📁 Found {len(json_files)} JSON files.\n")

for file_name in json_files:
	file_path = os.path.join(folder_path, file_name)

	try:
		df = pd.read_json(file_path)
	except Exception as e:
		print(f"⚠️ Error reading {file_name}: {e}")
		continue

	if 'ideal_steps' not in df.columns:
		print(f"⚠️ Skipping (no ideal_steps): {file_name}")
		continue

	total_rows = len(df)

	filtered_df = df[
		df['ideal_steps'].apply(
			lambda steps: not any(
				any(phrase in step for phrase in filter_list) for step in steps
			)
		)
	]

	filtered_rows = len(filtered_df)

	# Save filtered version (overwrite original JSON)
	# filtered_df.to_json(file_path, orient='records', indent=2)

	print(f"📌 {file_name} → Total: {total_rows} → Saved: {filtered_rows}")

print("\n✔ All applicable files updated.")